# Tier handoff: when to cross from the sharp law to the depth law

The single-tier benchmark in `minimizerStatistics.ipynb` answered a question that turned out to be the
wrong one. Neither contact law can do the whole job, so the free parameter worth measuring is not which
minimizer to use but when to cross between them.

**The depth law is invalid at deep overlap.** Past the medial-axis ridge its repulsion reverses sign and
bodies are pulled through, so a random start does not converge slowly, it converges confidently to a
stacked configuration that is a genuine force-balanced minimum. The monitor is `maximumDepth / inradius`,
which the law requires to be far below 1.

**The sharp law is only C1.** Its gradient jumps whenever a vertex crosses an edge. Measured on it, a
strong-Wolfe line search spends 17 to 35 force evaluations per step in some configurations against 2.04
in others, with no warning between the two regimes.

So the pipeline is sharp first, then depth, and this notebook measures the crossing. Three questions:

1. Does crossing on the validity ratio beat crossing at a fixed step count?
2. Which minimizer belongs on each tier?
3. How does the answer move with `N`?

## One measurement to read first

The validity ratio is confounded by folding, and in the direction that misleads. `inradius` is its
denominator, so a self-intersecting polygon reports a large ratio whether or not any pair is deeply
overlapped. Measured at N = 5 on a random seed, the ratio went 0.871 at the start to **0.998 after the
sharp tier had driven the overlap to zero** — it was reading the fold, not any penetration. Every table
below therefore carries `simple` beside the ratio, and the ratio is not interpretable without it.

In [ ]:
import json
import subprocess
import sys
import time

import numpy as np
from matplotlib import pyplot as plt

sys.path.insert(0, "tests")
import tierHandoff as th

# Fixed categorical identity, never cycled: the first three slots of a palette validated for
# colorblind separation on all pairs in both light and dark.
COLORS = {"fire": "#2a78d6", "lbfgs": "#eb6834", "cg": "#1baf7a"}
LABELS = {"fire": "FIRE", "lbfgs": "L-BFGS", "cg": "CG"}

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.axisbelow": True, "font.size": 9})

## Running one configuration

Each run executes in a separate process, for the reason established in the single-tier benchmark: the
sharp CUDA kernel raises `CUDA error 700` during line searches and leaves the context unusable, so a
failure caught in-process would return values computed after that point. A failed run is recorded as `crashed` and
the configuration is left unmeasured. There is no numpy fallback: it turned a fault costing
seconds into one costing the full per-cell timeout, and still produced no result.

In [ ]:
def runIsolated(spec, timeout = None):
    """`th.runOne(spec)` in a child process. Returns its record, or a `crashed` one."""
    command = [sys.executable, "tests/tierHandoff.py", "--spec", json.dumps(spec)]
    try:
        finished = subprocess.run(command, capture_output = True, text = True, timeout = timeout)
    except subprocess.TimeoutExpired:
        return dict(spec, outcome = "timeout", reached = False, seconds = float(timeout or 0.0),
                    maxForce = np.nan, energy = np.nan, evaluations = 0, steps = 0)
    for line in reversed(finished.stdout.splitlines()):
        if line.startswith("{"):
            return json.loads(line)
    return dict(spec, outcome = "crashed", reached = False, seconds = np.nan, maxForce = np.nan,
                energy = np.nan, evaluations = 0, steps = 0, stderr = finished.stderr[-600:])

def runCell(spec, timeout = 600):
    """One configuration, on the GPU. A kernel fault is recorded, not worked around.

    There is no numpy fallback. It converted a fault that costs seconds into one that costs the whole
    per-cell timeout and still produced nothing -- measured in the single-tier sweep, three cells spent
    1800 s each that way. A crashed cell is a result: it says this configuration cannot be run on the
    kernel as it stands."""
    record = runIsolated(dict(spec, device = "cuda"), timeout = timeout)
    record["device"] = "cuda"
    return record

def show(records, label = "schedule"):
    print(f"{label:<34} {'device':<7} {'outcome':<9} {'seconds':>9} {'steps':>7} {'evals':>8} "
          f"{'max|F|':>10} {'validity':>9} {'simple':>7}")
    for r in records:
        f = r.get("feasibility") or {}
        print(f"{str(r.get(label, r.get('plan', '?')))[:34]:<34} {r.get('device','?'):<7} "
              f"{r['outcome']:<9} {r.get('seconds', np.nan):>9.2f} {r.get('steps', 0):>7} "
              f"{r.get('evaluations', 0):>8} {r.get('maxForce', np.nan):>10.2e} "
              f"{r.get('validity', np.nan):>9.3f} {str(f.get('simple')):>7}")

## Parameters

In [ ]:
N = 6
n = 8
MODE = "springs"
PHI = 1.0
FORCE_TARGET = 1e-9
SHARP_BUDGET = 4000
DEPTH_BUDGET = 8000
TIMEOUT = 600

base = dict(N = N, n = n, mode = MODE, phi = PHI, forceTarget = FORCE_TARGET, place = "random")
print(f"N = {N}, n = {n}, {MODE}, phi = {PHI}")
print(f"validity limit for crossing: {th.VALIDITY_LIMIT} (the law wants dMax/rIn << 1)")

## 1. Where the run starts

The depth law's applicability at the seed, for both placements. `random` is the case it cannot start
from; `grid` is the overlap-free arrangement, included so a depth-only schedule can be run as a
control rather than assumed impossible.

In [ ]:
for place in ("random", "grid"):
    model = th.buildSystem(N = N, n = n, mode = MODE, phi = PHI, place = place)
    print(f"  {place:<7} validity {th.validity(model):>6.3f}   |turn| {th.turning(model):>7.1f}   "
          f"pair overlap {model.getPairOverlapArea():.4f}")
print(f"\n  crossing threshold {th.VALIDITY_LIMIT}: a seed above it is outside the depth law entirely.")

## 2. Crossing on a step count against crossing on the validity ratio

The same schedule with the crossing chosen two ways. The step-count rows fix how long the sharp tier
runs; the validity rows run it until `dMax/rIn` falls below a threshold, so the crossing adapts to the
system rather than being tuned per configuration.

A depth-only control is included. If it converges from a random start, the premise of the whole
notebook is wrong and should be discarded.

In [ ]:
schedules = []

# Control: straight to the depth tier from a random start.
schedules.append(("depth only", dict(base, plan = [["depth", "lbfgs", DEPTH_BUDGET]])))

# Crossing at a fixed step count.
for steps in (100, 500, 2000):
    schedules.append((f"sharp {steps} steps",
                      dict(base, plan = [["sharp", "lbfgs", steps],
                                         ["depth", "lbfgs", DEPTH_BUDGET]])))

# Crossing when the depth law becomes applicable.
for limit in (0.5, 0.25, 0.1):
    schedules.append((f"validity < {limit}",
                      dict(base, cross = limit,
                           plan = [["sharp", "lbfgs", SHARP_BUDGET],
                                   ["depth", "lbfgs", DEPTH_BUDGET]])))

crossing = []
for label, spec in schedules:
    record = runCell(spec, timeout = TIMEOUT)
    record["crossing"] = label
    crossing.append(record)
    stages = "  ".join(f"{s['tier']}/{s['minimizer']} {s['steps']}" for s in record.get("stages", []))
    print(f"  {label:<20} {record['outcome']:<9} {record.get('seconds', np.nan):>7.2f} s   {stages}")

show(crossing, label = "crossing")

## 3. Which minimizer on each tier

The sharp tier only has to make the state valid for the depth law, which is a much weaker requirement
than converging on it — and FIRE, which never converged on sharp in the single-tier benchmark, costs
about one force evaluation per step against a line search's two or more. So the sharp tier may want
FIRE precisely because it is not being asked to converge.

In [ ]:
pairs = []
for first in ("fire", "lbfgs"):
    for second in ("lbfgs", "cg"):
        spec = dict(base, cross = th.VALIDITY_LIMIT,
                    plan = [["sharp", first, SHARP_BUDGET], ["depth", second, DEPTH_BUDGET]])
        record = runCell(spec, timeout = TIMEOUT)
        record["pairing"] = f"sharp/{LABELS[first]} -> depth/{LABELS[second]}"
        pairs.append(record)
        print(f"  {record['pairing']:<32} {record['outcome']:<9} "
              f"{record.get('seconds', np.nan):>7.2f} s  {record.get('evaluations', 0):>7} evals")

show(pairs, label = "pairing")

In [ ]:
# Time by tier pairing, as a grouped comparison on one axis. Colour follows the DEPTH minimizer,
# since that is the series identity; the sharp choice is the category on the x-axis.
figure, axis = plt.subplots(figsize = (6.4, 3.6))
sharpChoices = ["fire", "lbfgs"]
width = 0.36
for offset, second in enumerate(("lbfgs", "cg")):
    heights = []
    for first in sharpChoices:
        match = [r for r in pairs if r["pairing"].startswith(f"sharp/{LABELS[first]}")
                 and r["pairing"].endswith(LABELS[second])]
        heights.append(match[0].get("seconds", np.nan) if match and match[0].get("reached")
                       else np.nan)
    axis.bar(np.arange(len(sharpChoices)) + offset * width, heights, width * 0.9,
             color = COLORS[second], label = f"depth: {LABELS[second]}")
axis.set_xticks(np.arange(len(sharpChoices)) + width / 2)
axis.set_xticklabels([f"sharp: {LABELS[c]}" for c in sharpChoices])
axis.set_ylabel("time to target (s)")
axis.set_title("tier pairing", loc = "left")
axis.legend(frameon = False)
figure.tight_layout()
plt.show()
print("missing bars are pairings that did not reach the target.")

## 4. How the crossing moves with N

Whether the best criterion is a constant or has to be computed per system. A validity threshold that
holds across `N` can be a fixed rule in a minimizer; a step count that drifts cannot.

In [ ]:
sweep = []
for count in (4, 6, 8):
    for label, extra in (("steps 500", dict(plan = [["sharp", "fire", 500],
                                                    ["depth", "lbfgs", DEPTH_BUDGET]])),
                         ("validity", dict(cross = th.VALIDITY_LIMIT,
                                           plan = [["sharp", "fire", SHARP_BUDGET],
                                                   ["depth", "lbfgs", DEPTH_BUDGET]]))):
        record = runCell(dict(base, N = count, **extra), timeout = TIMEOUT)
        record["criterion"] = label
        record["N"] = count
        sweep.append(record)
        crossed = record.get("stages", [{}])[0].get("steps", 0)
        print(f"  N = {count:<3} {label:<10} {record['outcome']:<9} "
              f"{record.get('seconds', np.nan):>7.2f} s   crossed after {crossed} sharp steps")

In [ ]:
figure, axis = plt.subplots(figsize = (6.4, 3.6))
for index, label in enumerate(("steps 500", "validity")):
    rows = sorted((r for r in sweep if r.get("criterion") == label), key = lambda r: r["N"])
    axis.plot([r["N"] for r in rows],
              [r["seconds"] if r.get("reached") else np.nan for r in rows],
              "o-", linewidth = 2, markersize = 7,
              color = list(COLORS.values())[index], label = label)
axis.set_xlabel("N polygons")
axis.set_ylabel("time to target (s)")
axis.set_title("crossing criterion against system size", loc = "left")
axis.legend(frameon = False)
figure.tight_layout()
plt.show()

## 5. Feasibility

What the timings are allowed to claim. `simple` is the guard the validity ratio needs beside it: a
False row means the ratio is partly or wholly reading a fold rather than a penetration.

In [ ]:
print(f"{'run':<34} {'|turn|':>9} {'simple':>7} {'validity':>9} {'overlap':>11} {'area err':>10}")
for record in crossing + pairs + sweep:
    f = record.get("feasibility") or {}
    if not f:
        print(f"{str(record.get('crossing', record.get('pairing', record.get('criterion'))))[:34]:<34} "
              f"  (none -- {record['outcome']})")
        continue
    name = record.get("crossing") or record.get("pairing") or f"N={record.get('N')} {record.get('criterion')}"
    print(f"{str(name)[:34]:<34} {f['turning']:>9.1f} {str(f['simple']):>7} "
          f"{record.get('validity', np.nan):>9.3f} {f['pairOverlap']:>11.2e} "
          f"{f['worstAreaError']:>10.2e}")

## Notes

Fill in from the tables above.

* whether the depth-only control converged. If it did, the two-tier premise is wrong.
* whether a validity threshold beats a step count, and whether the same threshold works across `N`.
* which minimizer the sharp tier wants, given that it only has to reach validity rather than converge.
* how many rows are `simple`. At `n >= 8` the single-tier sweep folded everywhere, and a folded run
  reports a validity ratio that is not about penetration.